In [6]:
import pandas as pd
train_df = pd.read_csv("train.csv")
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [7]:
test_df = pd.read_csv("test.csv")
print(train_df.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [8]:
test_df.dtypes

PassengerId      int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

In [9]:
num_cols=['Age','Fare','Pclass','SibSp','Parch']
cat_cols=['Sex','Embarked']
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
cat_pipeline=Pipeline(steps=[
    ('impute',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore')),
])
num_pipeline=Pipeline(steps=[
    ('impute',SimpleImputer(strategy='median')),
])
from sklearn.compose import ColumnTransformer
preprocessor=ColumnTransformer(
    transformers=[
        ('num',num_pipeline,num_cols),
        ('cat',cat_pipeline,cat_cols),
    ]
)

In [11]:
from sklearn.ensemble import RandomForestClassifier
X=train_df[num_cols+cat_cols]
y=train_df.Survived
my_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',RandomForestClassifier(n_estimators=100,random_state=1)),
])
my_pipeline.fit(X,y)
print(f"Score : {my_pipeline.score(X, y) * 100:.2f}%")


Score : 97.98%


In [13]:
from sklearn.model_selection import train_test_split
X_train ,X_valid ,y_train ,y_valid =train_test_split(X,y,test_size=0.2,random_state=0)
my_pipeline.fit(X_train,y_train)
print(f"Score : {my_pipeline.score(X_valid,y_valid) * 100:.2f}%")

Score : 83.80%


In [22]:
from sklearn.model_selection import GridSearchCV
param_grid={
    'model__n_estimators':[50,100,200],
    'model__max_depth':[5 ,10 ,None],
    'model__min_samples_split':[2,5,10],
}
grid_search=GridSearchCV(my_pipeline,param_grid,cv=5,scoring='accuracy')
grid_search.fit(X_train,y_train)
print(f"Les meilleurs reglages trouvés sont : {grid_search.best_params_}")
print(f"Nouveau score : {grid_search.score(X_valid, y_valid) * 100:.2f}%")

Les meilleurs reglages trouvés sont : {'model__max_depth': 10, 'model__min_samples_split': 10, 'model__n_estimators': 50}
Nouveau score : 83.24%


In [26]:
from xgboost import XGBClassifier
xgb_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',XGBClassifier(random_state=0)),
])
xgb_pipeline.fit(X_train,y_train)
print(f"le score finale est : {xgb_pipeline.score(X_valid,y_valid) *100:.2f}%")

le score finale est : 84.92%


In [29]:
X_train_prep=preprocessor.fit_transform(X_train)
X_valid_prep=preprocessor.transform(X_valid)
X_test_prep=preprocessor.transform(test_df)

In [31]:
xgb_model=XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    early_stopping_rounds=20,
    eval_metrics="logloss",
    random_state=0,
)
xgb_model.fit(
    X_train_prep,
    y_train,
    eval_set=[(X_valid_prep,y_valid)],
    verbose=True,
)

[0]	validation_0-logloss:0.65257
[1]	validation_0-logloss:0.63915
[2]	validation_0-logloss:0.62655
[3]	validation_0-logloss:0.61489
[4]	validation_0-logloss:0.60396
[5]	validation_0-logloss:0.59290
[6]	validation_0-logloss:0.58251
[7]	validation_0-logloss:0.57312
[8]	validation_0-logloss:0.56361
[9]	validation_0-logloss:0.55479
[10]	validation_0-logloss:0.54623
[11]	validation_0-logloss:0.53865
[12]	validation_0-logloss:0.53120
[13]	validation_0-logloss:0.52449
[14]	validation_0-logloss:0.51748
[15]	validation_0-logloss:0.51096
[16]	validation_0-logloss:0.50510
[17]	validation_0-logloss:0.49936
[18]	validation_0-logloss:0.49400
[19]	validation_0-logloss:0.48864
[20]	validation_0-logloss:0.48379
[21]	validation_0-logloss:0.47972
[22]	validation_0-logloss:0.47521
[23]	validation_0-logloss:0.47142
[24]	validation_0-logloss:0.46705
[25]	validation_0-logloss:0.46345
[26]	validation_0-logloss:0.45998
[27]	validation_0-logloss:0.45640
[28]	validation_0-logloss:0.45341
[29]	validation_0-loglos

c:\Users\LENOVO\anaconda3\Lib\site-packages\xgboost\callback.py:385: UserWarning: [18:56:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "eval_metrics" } are not used.

  self.starting_round = model.num_boosted_rounds()


[31]	validation_0-logloss:0.44489
[32]	validation_0-logloss:0.44176
[33]	validation_0-logloss:0.43887
[34]	validation_0-logloss:0.43664
[35]	validation_0-logloss:0.43425
[36]	validation_0-logloss:0.43229
[37]	validation_0-logloss:0.42984
[38]	validation_0-logloss:0.42789
[39]	validation_0-logloss:0.42606
[40]	validation_0-logloss:0.42395
[41]	validation_0-logloss:0.42207
[42]	validation_0-logloss:0.42018
[43]	validation_0-logloss:0.41868
[44]	validation_0-logloss:0.41690
[45]	validation_0-logloss:0.41513
[46]	validation_0-logloss:0.41374
[47]	validation_0-logloss:0.41218
[48]	validation_0-logloss:0.41054
[49]	validation_0-logloss:0.40868
[50]	validation_0-logloss:0.40760
[51]	validation_0-logloss:0.40629
[52]	validation_0-logloss:0.40473
[53]	validation_0-logloss:0.40344
[54]	validation_0-logloss:0.40221
[55]	validation_0-logloss:0.40101
[56]	validation_0-logloss:0.39997
[57]	validation_0-logloss:0.39872
[58]	validation_0-logloss:0.39748
[59]	validation_0-logloss:0.39647
[60]	validatio

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,20
,enable_categorical,True
,eval_metric,None


In [32]:
predicts_es = xgb_model.predict(X_test_prep)
submission_es = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': predicts_es
})
submission_es.to_csv('submission.csv', index=False)
print("Nouveau fichier submission.csv généré avec succès !")

Nouveau fichier submission.csv généré avec succès !


In [27]:
predicts=xgb_pipeline.predict(test_df)
output=pd.DataFrame({'PassengerId':test_df['PassengerId'],'Survived':predicts})
output.to_csv('submission.csv',index=False)